In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/pinterest_pipeline"
INBOX_PATH = f"{BASE}/inbox"
SCHEMA_LOC = f"{BASE}/_schema/pinterest_posts_bronze"
CHECKPOINT = f"{BASE}/_checkpoints/pinterest_posts_bronze"

BRONZE_TABLE = "workspace.default.pinterest_posts_bronze"

df_raw = (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", SCHEMA_LOC)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")

      # Key: don't fail the stream on bad lines / malformed JSON
      .option("mode", "PERMISSIVE")
      .option("columnNameOfCorruptRecord", "_corrupt_record")

      # Also capture any unexpected fields / parse issues
      .option("cloudFiles.rescuedDataColumn", "_rescued_data")

      # If your json is NOT multi-line (yours looks line-delimited), keep this false.
      .option("multiLine", "false")

      .load(INBOX_PATH)
)

# Helper to convert literal "null" (string) -> real NULL
def nullify_string(colname: str):
    return F.when(F.col(colname).isNull(), None) \
            .when(F.lower(F.col(colname)) == F.lit("null"), None) \
            .otherwise(F.col(colname))

df = (
    df_raw
    .withColumn("source_file_path", F.col("_metadata.file_path"))
    .withColumn("post_year", F.regexp_extract("source_file_path", r"post_year=(\d+)", 1).cast("int"))
    .withColumn("post_quarter", F.regexp_extract("source_file_path", r"post_quarter=(\d+)", 1).cast("int"))
    .withColumn("_ingest_ts", F.current_timestamp())

    # Normalize string "null" fields you know about
    .withColumn("source", nullify_string("source"))
    .withColumn("comments", nullify_string("comments"))

    # (Optional) If hashtags sometimes is literally "null" as a string:
    .withColumn("hashtags", nullify_string("hashtags"))

    # Parse date_posted if it arrives as string (it will in JSON)
    .withColumn("date_posted", F.to_timestamp("date_posted"))
)

(
  df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable(BRONZE_TABLE)
)